# Shikibu 評価関数の構造の実験室（Colab 用）

評価関数の**構造**（特徴・幅・層）を変えて学習し、同じデータ・同じ学習量で**検証の損失（val_loss）**を比べ、
良かったものを**やねうら王・Nuenagon で使える nn.bin**（やねうら王と同じ構造のもの）として書き出します。

流れは次の 4 段です。

1. **構造を選ぶ**（節 2・5）：比べる構造の名前を並べる。
2. **学習する**（節 6・7）：構造ごとに同じ条件で、裏で順番に学習する。
3. **比べる**（節 8・9）：検証の損失（val_loss）を並べる。
4. **書き出す**（節 10）：選んだ構造を nn.bin（または .shk）にする。

**学習には GPU のランタイムを使ってください**（メニュー「ランタイム」→「ランタイムのタイプを変更」→ GPU）。
学習は裏で続くので、表示のセルを止めても、ブラウザを再読み込みしても止まりません。途中で切れても、同じ手順で続きから再開できます。

## 1. Google Drive をマウントする

学習データ・リポジトリ・出力を Drive に置くため、最初にマウントします。

In [ ]:
# Google Drive を /content/drive にマウントする（Colab でだけ使える）
from google.colab import drive
drive.mount('/content/drive')

## 2. 設定（ここだけ書き換える）

`<…>` の目印を、自分の環境の値に書き換えてください。目印が残っていると、節 6 の設定の確認でエラーになります。

- `REPO_URL`・`BRANCH`：リポジトリの取得元。Drive に `REPO` が無いときだけ使います。
- `REPO`：Drive 上のリポジトリの場所。
- `WHEEL_DIR`：`shk_feat`（特徴抽出の部品）の wheel の保存先。
- `TRAIN_GLOB`：学習用の PSV のパス（`*.psv` のように書く）。
- `VAL_PATH`：検証用の PSV（1 ファイル）。学習には使わず、val_loss をここで測る。
- `OUT_ROOT`・`LAB_NAME`：出力は `OUT_ROOT/LAB_NAME/<構造の名前>` に作られる。
- `CACHE_DIR`：Colab のローカルの一時コピー先（Drive には置かない）。
- `STRUCTURES`：比べる構造の名前の一覧。行を増やしても、消してもかまいません（節 5 で意味を確かめられます）。
- `MAX_STEPS`・`BATCH_SIZE`・`VAL_EVERY`：**構造以外の条件**。比べる構造はすべて同じ値で学習します。

In [ ]:
# ===== 設定：<…> の目印を、自分の環境の値に書き換える =====
REPO_URL = "https://github.com/<GitHub のユーザー名>/Shikibu.git"    # リポジトリの取得元（Drive に無いときだけ使う）
BRANCH = "main"                                                     # 取得するブランチ
REPO = "/content/drive/MyDrive/<リポジトリを置くフォルダ>/Shikibu"      # Drive 上のリポジトリの場所
WHEEL_DIR = "/content/drive/MyDrive/<wheel を置くフォルダ>"              # shk_feat の wheel の保存先
TRAIN_GLOB = "/content/drive/MyDrive/<学習データのフォルダ>/*.psv"       # 学習用の PSV
VAL_PATH = "/content/drive/MyDrive/<検証データのフォルダ>/<検証用>.psv"   # 検証用の PSV（学習には使わない）
OUT_ROOT = "/content/drive/MyDrive/<出力のフォルダ>"                      # 実験の出力の親フォルダ
LAB_NAME = "<実験の名前>"                                                # 出力は OUT_ROOT/LAB_NAME/<構造の名前>
CACHE_DIR = "/content/shk_cache"                                      # Colab のローカルの一時コピー先

# 比べる構造（節 5 で、特徴の次元・書き出せるかを確かめられる）
STRUCTURES = [
    "raypair0_128x2-16-16",        # 今の RayPair_v0（基準）
    "halfkp_256x2-32-32_f",        # やねうら王の標準の HalfKP
    "halfkahm2_512x2-16-32_f",     # HalfKA_hm2 で、幅を 512 にしたもの
]

# 構造以外は全部同じにする（比べるための条件）
MAX_STEPS = 20000      # 学習の更新回数
BATCH_SIZE = 65536     # 1 回の更新に使う局面数
VAL_EVERY = 1000       # 何回の更新ごとに検証するか（val_loss を記録する間隔）

## 3. リポジトリを取得（または更新）する

`REPO` があれば `git pull` で更新し、無ければ `REPO_URL` から `git clone` します。

In [ ]:
import os
import subprocess

os.makedirs(WHEEL_DIR, exist_ok=True)
if os.path.isdir(os.path.join(REPO, ".git")):
    subprocess.run(["git", "-C", REPO, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO, "pull", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO], check=True)
print("リポジトリ:", REPO, "/ ブランチ:", BRANCH)

## 4. wheel（`shikibu_feat`）をインストールする

- `WHEEL_DIR` に `shk_feat-*.whl` があれば、それを入れます（数秒）。
- 無ければ Rust を入れて `maturin build --release` で wheel を作り、`WHEEL_DIR` に保存してから入れます（初回だけ）。
- Rust のソース（`crates/shk_feat`）を変えたときは、`WHEEL_DIR` の wheel を消してから実行し直してください。

In [ ]:
import glob

wheels = sorted(glob.glob(os.path.join(WHEEL_DIR, "shk_feat-*.whl")), key=os.path.getmtime)
if wheels:
    subprocess.run(["pip", "install", "-q", "--force-reinstall", wheels[-1]], check=True)
    print("保存済みの wheel を入れました:", wheels[-1])
else:
    if subprocess.run(["bash", "-lc", "command -v cargo"], capture_output=True).returncode != 0:
        subprocess.run("curl -sSf https://sh.rustup.rs | sh -s -- -y", shell=True, check=True)
    cargo_env = os.path.expanduser("~/.cargo/bin")
    env = dict(os.environ, PATH=cargo_env + os.pathsep + os.environ["PATH"])
    subprocess.run(["pip", "install", "-q", "maturin"], check=True)
    subprocess.run(["maturin", "build", "--release", "-m", "crates/shk_feat/Cargo.toml", "-o", WHEEL_DIR],
                   cwd=REPO, env=env, check=True)
    built = sorted(glob.glob(os.path.join(WHEEL_DIR, "shk_feat-*.whl")), key=os.path.getmtime)
    subprocess.run(["pip", "install", "-q", "--force-reinstall", built[-1]], check=True)
    print("wheel を作って保存しました:", built[-1])

## 5. 構造を選ぶ：何が違うか

構造の名前は `<特徴>_<幅 W>x2-<第2・第3層の大きさ L1-L2>[_オプション]` と読みます（例：`halfkp_256x2-32-32_f`）。

| 選ぶもの | 変わること | 目安 |
|---|---|---|
| **特徴**（`raypair0`・`halfkp`・`halfkahm2` など） | 盤のどこを、どう数えるか。HalfK 系は玉の位置も特徴に入れる。hm は左右の反転で数を減らす | やねうら王の形式（nn.bin）に書けるのは `halfk…` だけ |
| **幅 W**（`128`・`256`・`512`・`1024`） | 最初の層の幅。広いほど表現できるが、**重く、遅く、ファイルも大きくなる** | W を 2 倍にすると、最初の層の重みとメモリはおよそ 2 倍 |
| **層 L1-L2**（`32-32`・`16-32`・`8-32`） | 幅より後ろの層の大きさ | 小さいと軽いが、表現力は落ちやすい |
| **`_f`**（因数分解） | 学習のときだけ、玉に関係ない特徴を足す。学習が安定することがある | 書き出すときは元に畳み込むので、やねうら王の形式は変わらない |
| **`_psqt`・`_k3k3`**（独自） | 駒の点数の直接の項／玉の段で層を切り替える | まだ書き出せない。学習と比較だけできる |

下の表で、比べる構造の性質を確かめてください。`書き出し` が `nn.bin` なら、やねうら王・Nuenagon でそのまま使えます。

In [ ]:
import sys

sys.path.insert(0, os.path.join(REPO, "python"))
from shikibu import lab

print(f"{'構造の名前':34s} {'特徴の次元':>10s} {'重みの数':>12s} {'書き出し':>8s}  説明")
for name in STRUCTURES:
    d = lab.describe(name)
    fmt = d["export_format"] or "なし"
    print(f"{d['canonical']:34s} {d['train_features']:>10,d} {d['params']:>12,d} {fmt:>8s}  {lab.PRESETS.get(d['canonical'], '')}")

In [ ]:
# 選べる構造の一覧（説明つき）
for name, desc in lab.PRESETS.items():
    print(f"{name:32s} {desc}")

## 6. 同じ条件の設定を作る

構造の名前だけを変えて、ほかは全部同じ設定を作ります。出力は `OUT_ROOT/LAB_NAME/<構造の名前>` に分かれます。
ひな形（`configs/colab_lab.toml`）と同じ形の設定を、構造ごとに書き出します。

In [ ]:
from shikibu import train

COMMON = {
    "train.max_steps": MAX_STEPS,
    "train.total_steps": MAX_STEPS,
    "train.batch_size": BATCH_SIZE,
    "train.val_every_steps": VAL_EVERY,
    "data.cache_dir": CACHE_DIR,
}

LAB_DIR = os.path.join(OUT_ROOT, LAB_NAME)
CONFIG_PATHS = {}   # 構造の名前 → 設定ファイルのパス
for name in STRUCTURES:
    canon = lab.describe(name)["canonical"]
    out_dir = os.path.join(LAB_DIR, canon)
    os.makedirs(out_dir, exist_ok=True)
    cfg = lab.make_config(name, train_glob=TRAIN_GLOB, out_dir=out_dir, val_path=VAL_PATH, overrides=COMMON)
    path = os.path.join(out_dir, "config.toml")
    lab.write_config(cfg, path)
    CONFIG_PATHS[canon] = path

# 目印が残っていれば、ここで止まる（学習は始めない）
for path in CONFIG_PATHS.values():
    train.check_placeholders(train.load_config(path))
print("設定を書き出しました:")
for canon, path in CONFIG_PATHS.items():
    print(" ", canon, "→", path)

## 7. 学習を裏で順番に始める

構造ごとに、`shikibu.train` を順番に実行する手順を 1 つのスクリプトにして、裏で動かします。
同じ出力フォルダに `checkpoint.pt` があれば、自動で `--resume`（続きから）になります。

- 止めるときは、下のセルの `proc.terminate()` を実行します（最後の保存より後の分は失われます）。
- 学習の途中で同じセルを実行すると、二重に起動しないように確かめてから始めます。

In [ ]:
import shlex

env = dict(os.environ, PYTHONPATH=os.path.join(REPO, "python"))
if "proc" in globals() and proc.poll() is None:
    raise RuntimeError("学習が動いています。止めるなら proc.terminate() を実行してください。")

lines = ["set -u"]
for canon, path in CONFIG_PATHS.items():
    out_dir = os.path.dirname(path)
    resume = "--resume" if os.path.exists(os.path.join(out_dir, "checkpoint.pt")) else ""
    log = os.path.join(out_dir, "stdout.log")
    lines.append(f"python -u -m shikibu.train --config {shlex.quote(path)} {resume} >> {shlex.quote(log)} 2>&1")
RUN_SCRIPT = os.path.join(LAB_DIR, "run_all.sh")
with open(RUN_SCRIPT, "w", encoding="utf-8") as f:
    f.write("\n".join(lines) + "\n")
proc = subprocess.Popen(["bash", RUN_SCRIPT], cwd=REPO, env=env)
print("学習を裏で始めました（順番）:", list(CONFIG_PATHS))

## 8. 進み具合を見る

見たい構造の名前を `WATCH` に書いて実行します。15 秒ごとに、段階・進み具合・残り時間・学習曲線を表示します。
学習の間は、このセルを動かし続けてください（表示を止めても学習は止まりません）。

In [ ]:
sys.path.insert(0, os.path.join(REPO, "tools"))
import shk_monitor as mon

WATCH = "halfkp_256x2-32-32_f"   # 見たい構造の名前（節 6 の CONFIG_PATHS のキー）
mon.watch(CONFIG_PATHS[WATCH])

## 9. 比べる：検証の損失（val_loss）を並べる

全部の構造で検証が行われた、最後に共通の更新回数（`common_step`）の val_loss で並べます。
値が小さいほど、同じ局面をよく当てられていることを表します。

**注意**：val_loss は「局面をどれだけ当てられるか」の目安です。**強さは対局で確かめてください**。
比べるときは、構造以外（データ・検証データ・更新回数・バッチ・学習率・eval_scale・lam・seed）をそろえてください。

In [ ]:
ready = [os.path.dirname(p) for p in CONFIG_PATHS.values() if os.path.exists(os.path.join(os.path.dirname(p), "checkpoint.pt"))]
if not ready:
    print("まだ checkpoint.pt がありません。節 7 の学習が進むまで待ってください。")
else:
    rows = lab.compare_runs(ready)
    print(f"共通の更新回数: {rows[0]['common_step']}")
    print(f"{'構造の名前':34s} {'更新回数':>8s} {'val_loss（共通）':>16s} {'val_loss（最後）':>16s} {'eval_scale':>10s}")
    for r in rows:
        common = "—" if r["val_loss_at_common"] is None else f"{r['val_loss_at_common']:.5f}"
        last = "—" if r["val_loss"] is None else f"{r['val_loss']:.5f}"
        print(f"{r['network']:34s} {r['step']:>8d} {common:>16s} {last:>16s} {r['eval_scale']:>10.1f}")

## 10. 書き出す：やねうら王・Nuenagon で使う形にする

比べて良かった構造の名前を `ARCH` に書いて実行します（節 6 の `CONFIG_PATHS` のキーです）。
`halfk…` なら **nn.bin**、`raypair0・raypair1` なら **.shk** を `OUT_DIR/export/` に書き、読み直して確かめます。

- nn.bin のとき、表示される `FV_SCALE=<整数>` を、やねうら王・Nuenagon の設定 `FV_SCALE` に入れてください。
- 同じ行の `EDITION=<名前>` は、やねうら王をビルドするときの `YANEURAOU_EDITION` に使う名前です（`docs/DEVELOPMENT.md` も参照）。
- psqt・k3k3 の構造は書き出せません。その場合は理由が表示されます。

In [ ]:
ARCH = "halfkp_256x2-32-32_f"   # 書き出す構造の名前（節 6 の CONFIG_PATHS のキー）
OUT_DIR = os.path.dirname(CONFIG_PATHS[ARCH])
res = lab.export_run(OUT_DIR)
if res["path"] is None:
    print("書き出せません:", res["reason"])
else:
    print("書き出しました:", res["path"])
    print("形式:", res["format"])
    if res["format"] == "nn.bin":
        print(f"FV_SCALE={res['fv_scale']}")
        print(f"EDITION={res['yo_edition']}")

## 11. やねうら王・Nuenagon で使う（手順の要点）

1. 節 10 で書き出した `nn.bin` を、エンジンが評価関数を読むフォルダにコピーする。
   フォルダとファイル名は、エンジンの評価関数の設定と合わせる（使う版の説明は `README.md` の「エンジンで使う」を参照）。
2. 表示された `FV_SCALE` を、エンジンの設定 `FV_SCALE` に入れる。
3. `EDITION` の名前で、やねうら王をビルドする（例：`make YANEURAOU_EDITION=<EDITION の値> ...`）。
   ビルドしたバイナリで、対応する nn.bin を読ませる。
4. 比べた構造を、対局で確かめる（val_loss が良くても、強さは対局で決まる）。

Nuenagon の `.shk` は、`nuenagon.shk` をそのまま読みます（構造は RayPair_v0・RayPair_v1 だけ）。